# State · Node · Graph · Reducer

LangGraph 의 토대 4가지를 코드로 다룬다.

| 개념 | 한 줄 정의 |
|---|---|
| **State** | 그래프가 노드 사이를 오갈 때 들고 다니는 공유 데이터 |
| **Node** | State를 입력받아 새 State(또는 부분 dict)를 반환하는 함수 |
| **Edge** | 노드 간 흐름. `START` → ... → `END` |
| **Reducer** | 같은 State 필드를 여러 번 갱신할 때 "어떻게 합칠지" 규칙 |

> 이 노트북은 **LLM/API 키 없이** 돌아간다. 순수 그래프 메커니즘에 집중한다.

## 0. 환경 확인

In [ ]:
import sys
from importlib.metadata import version

print("python        :", sys.version.split()[0])
print("langgraph     :", version("langgraph"))
print("langchain-core:", version("langchain-core"))

## 1. State 스키마 — `TypedDict` vs Pydantic `BaseModel`

State는 그래프 전체가 공유하는 **데이터의 형태**다.
두 가지 방식이 가장 흔하다.

### 1-1. `TypedDict` — 가벼움, 런타임 검증 없음

구조만 선언한다. 타입과 다른 값이 들어와도 **런타임에서는 그냥 통과**한다 (정적 분석기/IDE에서만 잡힘).

> ### 📎 사전 지식: `TypedDict` / Pydantic `BaseModel` 이 뭔가?
>
> 둘 다 **"딕셔너리(dict)에 정해진 모양을 부여하는" 도구**다. 파이썬 `dict` 는 아무 키나 아무 타입이나 넣을 수 있어 자유롭지만, "이 데이터는 반드시 `id`(정수)와 `name`(문자)을 가져야 한다" 같은 약속을 강제하지 못한다. 그 약속을 미리 선언하는 것이 이 둘이다.
>
> | | `TypedDict` | Pydantic `BaseModel` |
> |---|---|---|
> | 출신 | **파이썬 표준** (`typing` 모듈, 설치 불필요) | **외부 라이브러리** (`pydantic`) |
> | 실체 | 그냥 dict | 클래스 인스턴스(객체) |
> | 값 접근 | `u["name"]` | `u.name` |
> | 잘못된 타입을 넣으면? | **그냥 통과** (IDE/타입검사기만 경고) | **런타임에 `ValidationError` 로 막음** |
> | 무게 | 매우 가벼움 | 약간 무거움 |
>
> → 빠른 프로토타이핑이면 `TypedDict`, 외부 입력 검증이 중요하면 Pydantic.
>
> #### `class User(TypedDict):` 의 소괄호는?
> 이 괄호는 **상속**을 뜻한다 (생성자 파라미터가 아니다). 파이썬에서 `class 이름(부모):` 형태의 괄호는 "어떤 부모 클래스를 물려받는가"를 적는 자리다.
>
> ```python
> class User(TypedDict):       # ← 정의할 때 괄호 = 상속 (TypedDict 를 부모로)
>     id: int                  #    본문의 id/name = 이 데이터가 가질 필드 선언
>     name: str
>
> u = User(id=1, name="철수")   # ← 만들 때 괄호 = 생성자 호출 (값 전달)
> ```
>
> 같은 `User` 라도 **정의할 때 괄호(상속)** 와 **만들 때 괄호(값 전달)** 는 완전히 다른 의미다. Pydantic 도 똑같이 `class User(BaseModel):` 로 `BaseModel` 을 상속받는다.

In [ ]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    email: str

u1: User = {"id": 1, "name": "starryeye", "email": "x@y.z"}
print(u1)

# 일부러 잘못된 타입 — 런타임 검증 없음
u2: User = {"id": 1, "name": 123, "email": "x@y.z"}  # type: ignore[typeddict-item]
print(u2, "  ← 그대로 통과")

### 1-2. Pydantic `BaseModel` — 런타임 검증 + 기본값/직렬화

선언이 비슷하지만 잘못된 타입이 오면 `ValidationError`.

In [ ]:
from pydantic import BaseModel, ValidationError

# BaseModel = 선언한 타입을 '실제로 런타임 검증'하는 클래스 (실체는 dict가 아니라 객체)
class UserPyd(BaseModel):
    id: int
    name: str
    email: str

# 생성은 키워드 인자로, 값 접근은 u.name (점 표기) — TypedDict의 u["name"]와 다름
print(UserPyd(id=1, name="starryeye", email="x@y.z"))

# 핵심: TypedDict와 달리 잘못된 타입은 ValidationError로 '막힌다'
try:
    UserPyd(id=1, name=123, email="x@y.z")   # name이 int → 검증 실패
except ValidationError as e:
    print("\nValidationError 발생 ↓")
    print(e)

**선택 가이드**

| 비교 | TypedDict | Pydantic BaseModel |
|---|---|---|
| 런타임 타입 검증 | ❌ | ✅ |
| 기본값 선언 | 어색함 | 자연스러움 |
| 직렬화/역직렬화 | 수동 | 내장 |
| 오버헤드 | 거의 0 | 약간 있음 |
| 노드 안에서 접근 | `state['name']` | `state.name` |

빠른 프로토타이핑이면 `TypedDict`, 외부 입력 검증이 필요하면 Pydantic.

## 2. State를 그래프의 "스키마" 로 지정하기

에이전트가 가져야 할 정보를 State 안에 정의하고,
노드들이 그 State 를 입력받아 부분 업데이트를 돌려주면 LangGraph 가 머지해준다.

더 나아가 **입력 스키마 / 출력 스키마를 분리**할 수도 있다.
→ `StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)`

이렇게 하면 외부에서 받는 키와 외부로 노출되는 키를 깨끗하게 분리할 수 있다.

In [ ]:
from langgraph.graph import StateGraph, START, END
# START = 그래프 진입점(가상 노드), END = 종료점(가상 노드)

# 입력용/출력용 스키마를 따로 두고, 내부 처리용은 둘을 합쳐서 쓸 수 있다
class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class OverallState(InputState, OutputState):
    """내부에서는 둘 다 들고 다닌다."""
    pass

# Node: 현재 State를 받아 '바뀔 필드만' dict로 반환 (전체를 다시 만들 필요 없음)
def answer_node(state: InputState):
    return {"answer": "bye", "question": state["question"]}

# input_schema/output_schema: 외부에 노출할 입력·출력 키를 제한
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(answer_node)              # 함수명을 노드 이름으로 자동 사용
builder.add_edge(START, "answer_node")     # 진입점 → 노드 (Edge = 실행 흐름)
builder.add_edge("answer_node", END)       # 노드 → 종료점
graph = builder.compile()                  # 실행 가능한 그래프로 컴파일

# 입력은 InputState 키(question)만 받고, 결과는 OutputState 키(answer)만 반환
graph.invoke({"question": "hi"})

컴파일된 그래프 객체를 `display` 하거나 `get_graph().draw_mermaid()` 로 구조를 볼 수 있다.

In [ ]:
print(graph.get_graph().draw_mermaid())

## 3. Reducer — 같은 필드를 "덮어쓰기" 가 아니라 "합치기"

노드가 State 의 어떤 필드를 반환할 때, **기본 동작은 덮어쓰기**다.
여러 노드가 같은 필드를 차례로(혹은 병렬로) 갱신하려면 "합치는 방법"을 명시해야 한다.
→ `Annotated[T, reducer]` 로 선언한다.

네 가지 케이스를 비교해보자.

### 3-1. Reducer 지정 안 함 — 덮어쓰기

n1 이 `["a"]` 를 반환하고 n2 가 `["b"]` 를 반환하면 → 최종 `["b"]`. n1 결과는 사라진다.

In [ ]:
# reducer를 지정하지 않으면 → 기본 동작은 '덮어쓰기(overwrite)'
class S1(TypedDict):
    items: list[str]      # reducer 없음

def n1(s): return {"items": ["a"]}
def n2(s): return {"items": ["b"]}   # n1이 넣은 ["a"]를 ["b"]로 덮어씀

gb = StateGraph(S1)
gb.add_node("n1", n1); gb.add_node("n2", n2)
gb.add_edge(START, "n1"); gb.add_edge("n1", "n2"); gb.add_edge("n2", END)
gb.compile().invoke({"items": []})   # 결과: {'items': ['b']} — ['a']는 사라짐

### 3-2. `operator.add` — 리스트/숫자 이어붙이기

리스트라면 concat, 숫자라면 합. `+` 연산자가 정의된 모든 타입에 동작.

In [ ]:
from typing import Annotated
from operator import add   # add(list, list) = 두 리스트 이어붙이기

class S2(TypedDict):
    # Annotated[타입, reducer]: 이 필드는 덮어쓰기 대신 reducer로 병합하라는 표시
    # add → 새 리스트를 기존 리스트 뒤에 이어붙임(누적)
    items: Annotated[list[str], add]

gb = StateGraph(S2)
gb.add_node("n1", n1); gb.add_node("n2", n2)
gb.add_edge(START, "n1"); gb.add_edge("n1", "n2"); gb.add_edge("n2", END)
gb.compile().invoke({"items": []})   # 결과: {'items': ['a', 'b']} — 누적됨

### 3-3. Custom reducer — 직접 정의한 함수

Reducer 시그니처: `(left, right) -> merged`. 어떤 합치기 규칙도 만들 수 있다.

In [ ]:
def merge_unique(left: list[str], right: list[str]) -> list[str]:
    # 중복 제거하며 누적 (순서 보존)
    seen = set(left)
    return left + [x for x in right if not (x in seen or seen.add(x))]

class S3(TypedDict):
    items: Annotated[list[str], merge_unique]

def n_ab(s): return {"items": ["a", "b"]}
def n_bc(s): return {"items": ["b", "c"]}

gb = StateGraph(S3)
gb.add_node("n_ab", n_ab); gb.add_node("n_bc", n_bc)
gb.add_edge(START, "n_ab"); gb.add_edge("n_ab", "n_bc"); gb.add_edge("n_bc", END)
gb.compile().invoke({"items": []})  # ['a', 'b', 'c']

### 3-4. `add_messages` — 메시지 리스트 전용

`langgraph.graph.message.add_messages` 는 메시지 누적용으로 특화된 리듀서.
- 새 메시지를 뒤에 append
- 같은 `id` 를 가진 메시지가 들어오면 **덮어쓰기 (update)**
- dict 형태(`{"role": ..., "content": ...}`) 입력도 자동으로 적절한 메시지 객체로 변환

In [ ]:
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    # add_messages = 메시지 리스트 전용 reducer (대화 누적의 표준)
    # 단순 이어붙이기에 더해, 같은 id 메시지는 중복 제거/교체까지 처리
    messages: Annotated[list[AnyMessage], add_messages]

def echo(state: ChatState):
    last = state["messages"][-1].content   # 마지막 메시지 내용
    return {"messages": [AIMessage(f"Echo: {last}")]}   # 새 메시지만 반환 → 누적됨

gb = StateGraph(ChatState)
gb.add_node("echo", echo)
gb.add_edge(START, "echo"); gb.add_edge("echo", END)
g = gb.compile()

result = g.invoke({"messages": [HumanMessage("ping")]})
for m in result["messages"]:
    print(f"{type(m).__name__:14s} | {m.content}")

## 4. Node 등록하는 두 가지 방법

```python
# 방법 1) 노드 이름을 명시
builder.add_node("my_name", my_fn)

# 방법 2) 함수명을 그대로 노드 이름으로
builder.add_node(my_fn)              # 노드 이름 = "my_fn"
```

엣지 연결 시 노드 이름이 일치해야 하므로 두 방식 중 하나로 일관되게 가는 것이 좋다.

In [ ]:
class S(TypedDict):
    messages: Annotated[list[str], add]

def chatbot(state: S):
    answer = "안녕하세요! 무엇을 도와드릴까요?"
    print("Answer :", answer)
    return {"messages": [answer]}

gb = StateGraph(S)
gb.add_node("chatbot", chatbot)        # 방법 1
gb.add_edge(START, "chatbot")
gb.add_edge("chatbot", END)
g = gb.compile()
g.invoke({"messages": ["안녕!"]})

## 정리

- **State** = 공유 데이터 스키마 (`TypedDict` / Pydantic). 검증이 필요하면 Pydantic.
- **입출력 분리** = `StateGraph(OverallState, input_schema=..., output_schema=...)` 로 외부 인터페이스만 노출.
- **Node** = `(state) -> dict` 함수. `add_node("name", fn)` 또는 `add_node(fn)`.
- **Edge** = `add_edge(src, dst)`. 흐름의 시작은 `START`, 종료는 `END`.
- **Reducer** = `Annotated[T, reducer]` 로 합치기 규칙 지정.
  - 미지정: 덮어쓰기
  - `operator.add`: concat / 합
  - custom: 원하는 어떤 규칙도
  - `add_messages`: 메시지 리스트 전용 (id 기반 dedup·update 지원)

다음: `02_messages_and_streaming.ipynb` — 메시지 상태 본격 활용 + 실행 결과를 받아오는 다양한 방법(invoke/ainvoke/stream/astream).